Extraindo PDF

In [1]:
!pip install transformers datasets pdfplumber accelerate sentencepiece


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
import re
import pdfplumber
from pathlib import Path
# from PyPDF2 import PdfReader
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM

c:\Users\rayco\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:

def extract_text_from_pdf(pdf_path):
    """Extrai texto de um arquivo PDF."""
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                pages.append(page_text.strip())
    return "\n\n".join(pages)

# Substitua pelo caminho do seu PDF
pdf_path = "20260018701.pdf"
full_text = extract_text_from_pdf(pdf_path)
print(f"Total de caracteres extraídos: {len(full_text)}")
print("\n--- INÍCIO DO TEXTO ---\n")
print(full_text[:500])

Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


Total de caracteres extraídos: 248714

--- INÍCIO DO TEXTO ---

Guia de Plantas
Medicinais de
Florianópolis

Estacartilhaéfrutodaexperiênciaedotrabalhodemuitas
pessoas. Sua publicação foi viabilizada pelo Projeto “Capaci-
taçãodeProfissionaisdaAtençãoBásicadeFlorianópolis”,do
Ministério da Saúde de 2013. Contribuíram para seu nasci-
mento profissionais e gestores da rede municipal de saúde
(SUSFlorianópolis);professores,técnicoseestudantesdaUni-
versidade Federal de Santa Catarina (UFSC); apoiadores do
HortoDidáticodePlantasMedicinaisdoHospitalUniversitário



Dividindo em chunks

In [ ]:
import re

def limpar_texto(texto):
    """
    Remove espaços extras e normaliza o texto bruto extraído do PDF.
    Isso ajuda a eliminar ruídos comuns em extrações de arquivos PDF.
    """
    # Substitui múltiplas quebras de linha ou espaços por um único espaço e remove espaços no início e no fim
    return re.sub(r'\s+', ' ', texto).strip()

def dividir_em_paragrafos(texto):
    """
    Divide o texto em parágrafos significativos.
    """
    # Dividindo os paragrafos por quebra de linha e limpando espaços extras
    paragrafos = texto.split("\n\n")
    # Filtro de ruído: ignora fragmentos menores que 50 caracteres (como números de página) 
    return [p.strip() for p in paragrafos if len(p.strip()) > 50]


In [ ]:
def chunk_text_especializado(paragrafos, max_chunk_chars=500, overlap_chars=100):
    """
    Agrupa parágrafos em blocos menores (chunks) com sobreposição.
    
    Sugestões aplicadas:
    - max_chunk_chars=500: Chunks menores tendem a ser mais focados.
    - overlap_chars: Evita a perda de contexto nas 'emendas' entre blocos.
    """
    chunks = []
    chunk_atual = ""

    for paragrafo in paragrafos:
        # TRATAMENTO DE INTEGRIDADE: 
        # Se um parágrafo sozinho for maior que o limite máximo, ele é dividido em partes.
        if len(paragrafo) > max_chunk_chars:
            if chunk_atual:
                chunks.append(chunk_atual)
                chunk_atual = ""
            # Divide o parágrafo gigante respeitando o overlap
            for i in range(0, len(paragrafo), max_chunk_chars - overlap_chars):
                chunks.append(paragrafo[i : i + max_chunk_chars])
            continue

        # Lógica de agrupamento com verificação de limite
        if len(chunk_atual) + len(paragrafo) + 2 <= max_chunk_chars:
            chunk_atual += ("\n\n" if chunk_atual else "") + paragrafo
        else:
            if chunk_atual:
                chunks.append(chunk_atual)
            
            # APLICAÇÃO DE SOBREPOSIÇÃO (Sliding Window):
            # O novo chunk começa com o final do chunk anterior para manter o contexto [2].
            inicio_com_contexto = chunk_atual[-overlap_chars:] if len(chunk_atual) > overlap_chars else ""
            chunk_atual = (inicio_com_contexto + " " if inicio_com_contexto else "") + paragrafo

    if chunk_atual:
        chunks.append(chunk_atual)

    return chunks


In [15]:
paragrafos = dividir_em_paragrafos(full_text)
print(f"\nParágrafos encontrados: {len(paragrafos)}")


Parágrafos encontrados: 154


In [16]:
chunks = chunk_text_especializado(paragrafos, max_chunk_chars=600)
print(f"Número de chunks: {len(chunks)}")
print(f"\nExemplo de chunk (primeiro):\n{chunks[0][:600]}...")

Número de chunks: 571

Exemplo de chunk (primeiro):
Estacartilhaéfrutodaexperiênciaedotrabalhodemuitas
pessoas. Sua publicação foi viabilizada pelo Projeto “Capaci-
taçãodeProfissionaisdaAtençãoBásicadeFlorianópolis”,do
Ministério da Saúde de 2013. Contribuíram para seu nasci-
mento profissionais e gestores da rede municipal de saúde
(SUSFlorianópolis);professores,técnicoseestudantesdaUni-
versidade Federal de Santa Catarina (UFSC); apoiadores do
HortoDidáticodePlantasMedicinaisdoHospitalUniversitário
da UFSC e sobretudo moradores das comunidades de Flori-
anópolis,quehámuitosséculosmantémvivaatradiçãoances-
tral da cura pelas plantas – compart...


Carregando modelo

In [43]:
# --- Modelo Seq2Seq ---
seq2seq_id = "google/flan-t5-base" 
seq2seq_tokenizer = AutoTokenizer.from_pretrained(seq2seq_id)
seq2seq_model = AutoModelForSeq2SeqLM.from_pretrained(seq2seq_id)

generator_seq2seq = pipeline(
    "text-generation",  # corrigido
    model=seq2seq_model,
    tokenizer=seq2seq_tokenizer,
)

print("Modelo carregado com sucesso.")

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 2259.49it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadMo

Modelo carregado com sucesso.


In [44]:
# template do prompt — fora do loop
PROMPT_SEQ2SEQ = """Você é um especialista em plantas medicinais.
Com base APENAS no texto abaixo, gere exatamente um objeto JSON com os campos:
- "instruction": pergunta que um usuário faria sobre o texto.
- "output": resposta baseada no texto.

Retorne apenas o JSON, sem explicações.

Texto:
{chunk}"""


In [45]:
# geração
triplas_seq2seq = []
for i, chunk in enumerate(clean_chunks):
    prompt = PROMPT_SEQ2SEQ.replace("{chunk}", chunk)  # corrigido
    try:
        result = run_model(generator_seq2seq, prompt)
        triple = extract_triple(result, chunk)
        if len(triple["output"]) > 20 and len(triple["instruction"]) > 5:
            triplas_seq2seq.append(triple)
        if i % 50 == 0:
            print(f"Processados {i}/{len(clean_chunks)} chunks (seq2seq)")
    except Exception as e:
        print(f"Erro no chunk {i}: {e}")

print(f"Triplas geradas (seq2seq): {len(triplas_seq2seq)}")

Triplas geradas (seq2seq): 0


In [46]:
# testa num chunk só pra ver a saída bruta
chunk_teste = clean_chunks[0]
prompt_teste = PROMPT_SEQ2SEQ.replace("{chunk}", chunk_teste)

print("=== PROMPT ENVIADO ===")
print(prompt_teste[:300])
print("\n=== SAÍDA BRUTA DO MODELO ===")
saida = run_model(generator_seq2seq, prompt_teste)
print(repr(saida))
print("\n=== SAÍDA FORMATADA ===")
print(saida)

IndexError: list index out of range

In [47]:
print(f"Total de chunks: {len(chunks)}")
print(f"Chunks após filtro: {len(clean_chunks)}")

# vê por que estão sendo filtrados
for i, chunk in enumerate(chunks[:5]):
    print(f"\n--- chunk {i} ---")
    print(f"tamanho: {len(chunk)}")
    print(f"palavras únicas: {len(set(chunk.split()))}")
    print(f"is_bad_chunk: {is_bad_chunk(chunk)}")
    print(chunk[:200])

Total de chunks: 154
Chunks após filtro: 0

--- chunk 0 ---
tamanho: 974
palavras únicas: 71
is_bad_chunk: False
Estacartilhaéfrutodaexperiênciaedotrabalhodemuitas
pessoas. Sua publicação foi viabilizada pelo Projeto “Capaci-
taçãodeProfissionaisdaAtençãoBásicadeFlorianópolis”,do
Ministério da Saúde de 2013. Con

--- chunk 1 ---
tamanho: 1166
palavras únicas: 60
is_bad_chunk: False
Colaboradores
Alésio dos Passos Santos: Bruxo morador da Lagoa da Con-
ceição,professordefitoterapia,ambientalista.
Amanda Ellen de Athayde: Programa de Pós graduação em
CiênciasFarmacêuticasdaUFSC.
A

--- chunk 2 ---
tamanho: 1139
palavras únicas: 66
is_bad_chunk: False
MaiqueWeberBiavatti:ProfessoraDoutoradoDepartamento
deCiênciasFarmacêuticasdaUFSC.
Mayara Krasinski Caddah: Professora Doutora do Departa-
mentodeBotânicadaUFSC.
MelissaCostaSantos:FarmacêuticadaPrefe

--- chunk 3 ---
tamanho: 1907
palavras únicas: 46
is_bad_chunk: False
Sumário
7...........Amedicinadasimplicidade
10.........Introdução
17...